# 26-Class ECG Classifier V3.2 (PTB-XL + Chapman + Challenge + CODE-15%)

Trains ECGNetJoint on **~457K records** with transfer learning from V3 best (AUROC=0.9682).

**Requires tars pre-built and synced to Drive (see CLAUDE.md).**

---
| Cell | Runtime | Time | What |
|------|---------|------|------|
| 1 | **GPU/TPU** | ~30-40 min | Mount Drive, copy scripts, restore all 4 datasets from tars |
| 2 | **GPU/TPU** | ~40-90 min (TPU) | Train V3.2 + save model to Drive |


In [ ]:
# -----------------------------------------------------------------------------
# Cell 1 -- GPU/TPU runtime | Setup + restore all data from Drive tars
# Tars must be pre-built and synced to Drive/EKG/ekg_datasets/:
#   code15.tar        (~68 GB)  challenge2021.tar (~7 GB)
#   chapman.tar       (~5.5 GB) ptbxl.tar         (~2.7 GB)
# Extracted largest-first so disk-space or sync errors surface immediately.
# Each tar includes its index file. Cell fails fast if any required tar missing.
# IMPORTANT: Do NOT import torch_xla here -- it locks /dev/vfio/0.
# -----------------------------------------------------------------------------
import time, os, sys, subprocess, shutil
from pathlib import Path
t0 = time.time()
print('=' * 60)
print('Cell 1: GPU/TPU setup + restore data')
print('=' * 60)

# -- Verify accelerator ---------------------------------------------------
import torch
if os.path.exists('/dev/vfio/0'):
    accel = 'TPU'
elif torch.cuda.is_available():
    accel = f'GPU ({torch.cuda.get_device_name(0)})'
else:
    raise RuntimeError('No GPU or TPU! Runtime > Change runtime type > T4 GPU or TPU v6e')
print(f'Accelerator: {accel}')

# TPU: install torch_xla if needed (via subprocess only -- never import it here)
if accel == 'TPU':
    if subprocess.run([sys.executable, '-c', 'import torch_xla'],
                      capture_output=True).returncode != 0:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        'torch_xla'], check=True)
        print('torch_xla installed')

# -- Mount Drive + install deps -------------------------------------------
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'wfdb', 'scipy', 'scikit-learn', 'h5py'], check=True)
print(f'Deps ready ({time.time()-t0:.0f}s)')

# -- Check available disk space -------------------------------------------
import shutil as _shutil
free_gb = _shutil.disk_usage('/content').free / 1e9
print(f'Local SSD free: {free_gb:.0f} GB (need ~85 GB total for all datasets)')
if free_gb < 85:
    print(f'WARNING: only {free_gb:.0f} GB free -- some datasets may not fit')

# -- Copy scripts from Drive ----------------------------------------------
SCRIPTS_DIR  = Path('/content/drive/MyDrive/EKG')
DRIVE_DATA   = Path('/content/drive/MyDrive/EKG/ekg_datasets')
DRIVE_MODELS = Path('/content/drive/MyDrive/EKG/models')
os.chdir('/content')
os.makedirs('ekg_datasets', exist_ok=True)
os.makedirs('models', exist_ok=True)

NEEDED = ['multilabel_v3.py', 'multilabel_classifier.py',
          'cnn_classifier.py', 'dataset_chapman.py', 'dataset_challenge.py',
          'dataset_code15.py']
for s in NEEDED:
    src = SCRIPTS_DIR / s
    if not src.exists():
        raise FileNotFoundError(f'{s} not found at {SCRIPTS_DIR}/')
    shutil.copy(src, f'/content/{s}')
print(f'Scripts copied ({time.time()-t0:.0f}s)')

# -- Restore datasets from Drive tars ------------------------------------
# Extracted largest-first: fail fast on disk-space / incomplete-sync errors.
#
# Tar path conventions:
#   code15.tar        -- paths: code15/...          extract: -C /content/ekg_datasets
#   challenge2021.tar -- paths: challenge2021/...    extract: -C /content/ekg_datasets
#   chapman.tar       -- paths: ekg_datasets/...     extract: -C /content
#   ptbxl.tar         -- paths: ptbxl/...            extract: -C /content/ekg_datasets

def extract_tar(tar_path: Path, target: str, label: str):
    size_gb = tar_path.stat().st_size / 1e9
    print(f'{label}: extracting {tar_path.name} ({size_gb:.1f} GB)...', flush=True)
    r = subprocess.run(['tar', 'xf', str(tar_path), '-C', target],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(
            f'{label}: tar failed (exit {r.returncode})\n'
            f'stderr: {r.stderr[:800]}\n'
            f'Possible causes: Drive sync not finished, corrupted tar, or disk full.'
        )
    print(f'{label}: done ({time.time()-t0:.0f}s)', flush=True)

# [1/4] CODE-15% (~68 GB, ~15-20 min) -- largest first
code15_dir = Path('/content/ekg_datasets/code15')
n_h5 = sum(1 for i in range(18)
           if (code15_dir / 'raw' / f'exams_part{i}.h5').exists())
if n_h5 == 18:
    print(f'CODE-15%: {n_h5}/18 H5 files already on SSD -- skipping')
else:
    tar = DRIVE_DATA / 'code15.tar'
    if not tar.exists():
        raise FileNotFoundError(
            f'code15.tar not found at {DRIVE_DATA}.\n'
            'Build locally: run run_code15_after_download.bat, then let Drive finish syncing.'
        )
    extract_tar(tar, '/content/ekg_datasets', 'CODE-15%')
    n_h5 = sum(1 for i in range(18)
               if (code15_dir / 'raw' / f'exams_part{i}.h5').exists())
    print(f'CODE-15%: {n_h5}/18 H5 files on SSD')

# [2/4] Challenge 2021 (~7 GB, ~10 min) -- optional
challenge_dir = Path('/content/ekg_datasets/challenge2021')
if len(list(challenge_dir.rglob('*.mat'))) >= 50000:
    print('Challenge: already on SSD -- skipping')
else:
    tar = DRIVE_DATA / 'challenge2021.tar'
    if tar.exists():
        extract_tar(tar, '/content/ekg_datasets', 'Challenge')
    else:
        print('WARNING: challenge2021.tar not found -- training on PTB-XL + Chapman + CODE-15% only')

# [3/4] Chapman (~5.5 GB, ~5 min)
chapman_dir = Path('/content/ekg_datasets/chapman')
if len(list(chapman_dir.rglob('*.mat'))) >= 44000:
    print('Chapman: already on SSD -- skipping')
else:
    tar = DRIVE_DATA / 'chapman.tar'
    if not tar.exists():
        raise FileNotFoundError(f'chapman.tar not found at {DRIVE_DATA}.')
    extract_tar(tar, '/content', 'Chapman')

# [4/4] PTB-XL (~2.7 GB, ~8 min)
ptbxl_dir = Path('/content/ekg_datasets/ptbxl')
if len(list(ptbxl_dir.rglob('*.dat'))) >= 18000:
    print('PTB-XL: already on SSD -- skipping')
else:
    tar = DRIVE_DATA / 'ptbxl.tar'
    if not tar.exists():
        raise FileNotFoundError(f'ptbxl.tar not found at {DRIVE_DATA}.')
    extract_tar(tar, '/content/ekg_datasets', 'PTB-XL')

# -- Restore v2 model (transfer learning) --------------------------------
v2_dst = Path('/content/models/ecg_multilabel_v2.pt')
if not v2_dst.exists():
    v2_src = DRIVE_MODELS / 'ecg_multilabel_v2.pt'
    if v2_src.exists():
        shutil.copy(v2_src, v2_dst)
        print('v2 model copied for transfer learning')
    else:
        print('WARNING: v2 model not found -- training from scratch')

print(f'\nCell 1 done ({time.time()-t0:.0f}s) -- ready to train')


In [ ]:
# -----------------------------------------------------------------------------
# Cell 2 -- GPU/TPU runtime | Train V3.2 + save to Drive
# TPU (v6e): ~40-90 min  |  GPU (T4): ~2-4 hrs
# First TPU epoch is slow (XLA compilation) -- this is normal.
# IMPORTANT: Do NOT import torch_xla -- it locks /dev/vfio/0.
# -----------------------------------------------------------------------------
import time, os, sys, subprocess, shutil
from pathlib import Path
t0 = time.time()
print('=' * 60)
print('Cell 2: Train V3.2 + save model')
print('=' * 60)
os.chdir('/content')

import torch
if os.path.exists('/dev/vfio/0'):
    accel = 'TPU'
elif torch.cuda.is_available():
    accel = f'GPU ({torch.cuda.get_device_name(0)})'
else:
    raise RuntimeError('No GPU or TPU! Run Cell 1 first.')

# TPU pre-flight: verify device not locked
if accel == 'TPU':
    try:
        fd = os.open('/dev/vfio/0', os.O_RDONLY | os.O_NONBLOCK)
        os.close(fd)
    except OSError as e:
        raise RuntimeError(
            f'TPU device busy ({e}). Runtime > Restart runtime, then re-run Cell 1 + 2.'
        )

print(f'Accelerator: {accel}')
print('Starting multilabel_v3.py...', flush=True)
print('-' * 60, flush=True)

proc = subprocess.Popen(
    [sys.executable, '-u', 'multilabel_v3.py'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

print('-' * 60, flush=True)
if proc.returncode != 0:
    raise RuntimeError(f'Training failed (exit {proc.returncode})')

# -- Save model to Drive --------------------------------------------------
src = Path('/content/models/ecg_multilabel_v3.pt')
DRIVE_MODELS = Path('/content/drive/MyDrive/EKG/models')
DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
if src.exists():
    dst = DRIVE_MODELS / 'ecg_multilabel_v3.pt'
    shutil.copy(src, dst)
    print(f'Model saved to Drive ({dst.stat().st_size/1e6:.0f} MB)')
    print('Syncs to local PC via Google Drive for Desktop.')
else:
    print('WARNING: model file not found')

print(f'\nCell 2 done ({time.time()-t0:.0f}s)')
